In [ ]:
from settings import Setting
from data_manager import DataManager
from model_factory import ModelFactory
from trainer import Trainer
import torch.nn as nn
import torch.optim as optim
import visualizer

setting = Setting() 
data_manager = DataManager(setting)

for name in setting.MODEL_NAME_LIST:
    print(f"\n--- {name} 모델 실험 시작 ---")
    
    # 모델과 전처리 규칙을 가져옴
    model, train_transform, val_transform = ModelFactory.create_model_and_transforms(name, 
                                                                data_manager.num_classes)
    
    # 가져온 전처리 규칙으로 데이터를 모델 학습용 및 평가용으로 가공한 객체인 로더를 가져옴
    train_loader, val_loader = data_manager.get_loaders(train_transform, val_transform)

    # 손실 함수는 획득한 결과와 실제 값 사이의 틀린 정도를 측정하는 함수
    # 학습 중에 이 값을 최소화하려고 하며, 예측과 정답을 비교해 손실을 계산
    # 모델 매개변수 최적화하기 - 파이토치 한국어 튜토리얼
    # https://tutorials.pytorch.kr/beginner/basics/optimization_tutorial.html
    criterion = nn.CrossEntropyLoss()
    
    # 옵티마이저는 손실 함수의 최저점을 찾아주는 탐색기

    # 가중치 재학습 여부를 False로 한 것들은 옵티마이저에 넣을 필요가 없음
    # 따라서 실제로 학습할 파라미터들만 모으는 리스트를 따로 생성
    # 이미 모델 생성 과정에서 마지막 레이어만 True가 되었으니 그것만 들어갈 예정
    param_list = [] 

    for param in model.parameters():
        if param.requires_grad == True:
            param_list.append(param) 
    
    # 컴퓨터 비전을 위한 전이 학습 튜토리얼에서는 옵티마이저로 SGD 사용
    # SGD 옵티마이저에 재학습 가능한 파라미터만 있는 리스트, 학습률, 모멘텀(관성) 전달해 객체 생성
    # optimizer = optim.SGD(param_list, lr=setting.LEARNING_RATE, momentum=setting.OPTIM_MOMENTUM)
    
    # 옵티마이저로 Adam을 쓴 버전, SGD와 바꿔가며 테스트
    optimizer = optim.Adam(param_list, lr=setting.LEARNING_RATE)

    # 학습용 엔진에 모델, 설정, 손실 함수, 옵티마이저를 전달해 객체 생성
    trainer = Trainer(model, setting, criterion, optimizer)

    # 인공지능 강의 #6 5장 딥러닝과 텐서플로를 참고
    # 텐서플로에서는 model.fit() 메서드가 학습 도중에 발생한 정보를 hist 객체에 저장해 둠
    # hist.history['accuracy']처럼 쓰기만 해도 바로 시각화에 쓸 수 있음
    # 그런데 파이토치에는 그런 기능이 없으므로 따로 딕셔너리 만들어 기록할 필요가 있음
    history = {
        'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []
    }

    # 정해진 횟수만큼 에포크 반복
    for epoch in range(setting.EPOCHS):
        print(f"\n[Epoch {epoch+1}/{setting.EPOCHS} 시작]")

        # 먼저 전체 학습 데이터셋에 대해 한 바퀴 훈련한 뒤 전체 평균 오차와 정확도 출력
        train_loss, train_acc = trainer.train_epoch(train_loader)
        print(f"[Train] Loss: {train_loss:.4f} | Acc: {train_acc * 100:.2f}%")

        # 그 다음 가중치 수정 없이 현재 모델 평가 후 평균 오차와 정확도 출력
        val_loss, val_acc = trainer.evaluate(val_loader)
        print(f"[ Val ] Loss: {val_loss:.4f} | Acc: {val_acc * 100:.2f}%")

        if setting.CAN_DRAW_PLOT:
            history['train_loss'].append(train_loss)
            # 정확도 객체는 파이토치의 텐서 객체로, 그대로 들어가면 충돌 또는 메모리 점유 위험 가능성 있음
            # 따라서 item()을 붙여 순수한 숫자 데이터로 바꿔 넣는 것이 안전
            history['train_acc'].append(train_acc.item())
            history['val_loss'].append(val_loss)
            history['val_acc'].append(val_acc.item())
    
    if setting.CAN_DRAW_PLOT:
        visualizer.draw_plot(history)

    print(f"\n--- {name} 모델 실험 끝 ---")

엔비디아 GPU cuda 사용

--- resnet50 모델 실험 시작 ---

[Epoch 1/5 시작]
[Train] Loss: 1.8695 | Acc: 27.36%
[ Val ] Loss: 1.8491 | Acc: 24.53%

[Epoch 2/5 시작]
[Train] Loss: 1.6394 | Acc: 44.26%
[ Val ] Loss: 1.7704 | Acc: 32.08%

[Epoch 3/5 시작]
[Train] Loss: 1.4864 | Acc: 49.66%
[ Val ] Loss: 1.7014 | Acc: 38.36%

[Epoch 4/5 시작]
[Train] Loss: 1.3849 | Acc: 54.73%
[ Val ] Loss: 1.6758 | Acc: 38.99%

[Epoch 5/5 시작]
[Train] Loss: 1.3079 | Acc: 60.47%
[ Val ] Loss: 1.6223 | Acc: 42.77%

--- resnet50 모델 실험 끝 ---

--- densenet121 모델 실험 시작 ---

[Epoch 1/5 시작]
[Train] Loss: 1.9213 | Acc: 26.35%
[ Val ] Loss: 1.9518 | Acc: 28.93%

[Epoch 2/5 시작]
[Train] Loss: 1.6576 | Acc: 40.88%
[ Val ] Loss: 1.7826 | Acc: 28.30%

[Epoch 3/5 시작]
[Train] Loss: 1.5015 | Acc: 47.64%
[ Val ] Loss: 1.8156 | Acc: 25.16%

[Epoch 4/5 시작]
[Train] Loss: 1.3928 | Acc: 47.64%
[ Val ] Loss: 1.7006 | Acc: 29.56%

[Epoch 5/5 시작]
[Train] Loss: 1.3629 | Acc: 56.76%
[ Val ] Loss: 1.6729 | Acc: 36.48%

--- densenet121 모델 실험 끝 ---
